# 08_fix_all_data.py
Comprehensive script to generate/verify ALL real data for the IEEE Access manuscript.
Fixes issues A-E identified during review:
  A. 8-seed attention per-seed metrics on full data
  B. SHAP bootstrap CIs (re-compute from raw SHAP values)
  C. Concentration index: correct computation on all 189K samples
  D. Operational regime stats with correct MC-dropout data
  E. Figure labels: rename "Gini coefficient" -> "Concentration index"

Run from the notebooks/ directory:
    python 08_fix_all_data.py

Estimated runtime:
  - Parts 1-4 (quick): ~5 minutes
  - Part 5 (8-seed attention training): ~4 hours on CPU

In [ ]:
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr

# Paths
BASE_DIR = Path(__file__).parent.parent
RESULTS_DIR = BASE_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
MODELS_DIR = RESULTS_DIR / 'models'

print("=" * 70)
print("IEEE Access Manuscript Data Verification & Fix Script")
print("=" * 70)

## PART 1: Re-compute operational regime stats (Issue D)

In [ ]:
def part1_regime_stats():
    """Re-compute regime-specific error stats using correct MC-dropout data."""
    print("\n" + "=" * 70)
    print("PART 1: Re-computing operational regime stats with real MC-dropout")
    print("=" * 70)

    # Load data
    X_test = np.load(RESULTS_DIR / 'X_test.npy')
    y_test = np.load(RESULTS_DIR / 'y_test.npy')
    y_test_mean = np.load(RESULTS_DIR / 'y_test_mean_mc_dropout.npy')
    y_test_std = np.load(RESULTS_DIR / 'y_test_std_mc_dropout.npy')

    import pickle
    with open(RESULTS_DIR / 'feature_names.pkl', 'rb') as f:
        feature_names = pickle.load(f)

    print(f"  Loaded X_test: {X_test.shape}")
    print(f"  Loaded y_test: {y_test.shape}")
    print(f"  Feature names: {feature_names}")
    print(f"  MC-dropout mean sigma: {y_test_std.mean():.6f} +/- {y_test_std.std():.6f}")
    print(f"  MC-dropout median sigma: {np.median(y_test_std):.6f}")
    print(f"  MC-dropout range: [{y_test_std.min():.6f}, {y_test_std.max():.6f}]")

    # Compute absolute errors
    abs_errors = np.abs(y_test - y_test_mean)

    # Extract last-timestep features (most recent reading)
    # feature_names = ['humidity', 'temperature', 'humiditysol', 'temperaturesol', 'co2', 'lumière']
    humidity_idx = feature_names.index('humidity')
    temp_idx = feature_names.index('temperature')
    soil_moisture_idx = feature_names.index('humiditysol')

    humidity_vals = X_test[:, -1, humidity_idx]
    temp_vals = X_test[:, -1, temp_idx]
    soil_moisture_vals = X_test[:, -1, soil_moisture_idx]

    # --- High valve commands (>0.6) vs low (<0.3) ---
    high_valve_mask = y_test > 0.6
    low_valve_mask = y_test < 0.3

    n_high = high_valve_mask.sum()
    n_low = low_valve_mask.sum()
    mae_high = abs_errors[high_valve_mask].mean()
    mae_low = abs_errors[low_valve_mask].mean()
    sigma_high = y_test_std[high_valve_mask].mean()
    sigma_low = y_test_std[low_valve_mask].mean()

    print(f"\n  --- Valve Command Regime Analysis ---")
    print(f"  High valve (>0.6): n={n_high:,}, MAE={mae_high:.4f}, mean sigma={sigma_high:.4f}")
    print(f"  Low valve (<0.3):  n={n_low:,}, MAE={mae_low:.4f}, mean sigma={sigma_low:.4f}")
    print(f"  MAE ratio (high/low): {mae_high/mae_low:.1f}x")
    print(f"  Sigma ratio (high/low): {sigma_high/sigma_low:.1f}x")

    # Cohen's d for valve regime difference
    pooled_std = np.sqrt((abs_errors[high_valve_mask].std()**2 + abs_errors[low_valve_mask].std()**2) / 2)
    cohens_d = (mae_high - mae_low) / pooled_std
    print(f"  Cohen's d: {cohens_d:.2f}")

    # --- Humid regimes (top quartile) ---
    humidity_q75 = np.percentile(humidity_vals, 75)
    humid_mask = humidity_vals > humidity_q75
    n_humid = humid_mask.sum()
    mae_humid = abs_errors[humid_mask].mean()
    std_humid = abs_errors[humid_mask].std()

    print(f"\n  --- Humidity Regime Analysis ---")
    print(f"  Top quartile humidity (>{humidity_q75:.2f}): n={n_humid:,}")
    print(f"  MAE (humid regime): {mae_humid:.4f}")
    print(f"  Error std (humid regime): {std_humid:.4f}")

    # --- Temperature extremes ---
    temp_extreme_mask = (temp_vals > 0.64) | (temp_vals < 0.3)  # Approximate normalized thresholds
    # We need to understand the normalization. Since data is MinMax [0,1], let's compute stats
    print(f"\n  --- Temperature Analysis ---")
    print(f"  Temperature range (normalized): [{temp_vals.min():.4f}, {temp_vals.max():.4f}]")
    print(f"  Temperature mean: {temp_vals.mean():.4f}")

    # Use actual quartiles for temperature analysis
    temp_q25 = np.percentile(temp_vals, 25)
    temp_q75 = np.percentile(temp_vals, 75)
    temp_low_mask = temp_vals < temp_q25
    temp_high_mask = temp_vals > temp_q75
    temp_mid_mask = (temp_vals >= temp_q25) & (temp_vals <= temp_q75)

    mae_temp_low = abs_errors[temp_low_mask].mean()
    mae_temp_high = abs_errors[temp_high_mask].mean()
    mae_temp_mid = abs_errors[temp_mid_mask].mean()
    mae_temp_extreme = abs_errors[temp_low_mask | temp_high_mask].mean()

    print(f"  Temperature Q1 (<{temp_q25:.3f}): MAE={mae_temp_low:.4f}, n={temp_low_mask.sum():,}")
    print(f"  Temperature mid ({temp_q25:.3f}-{temp_q75:.3f}): MAE={mae_temp_mid:.4f}, n={temp_mid_mask.sum():,}")
    print(f"  Temperature Q4 (>{temp_q75:.3f}): MAE={mae_temp_high:.4f}, n={temp_high_mask.sum():,}")
    print(f"  Extreme vs mid MAE ratio: {mae_temp_extreme/mae_temp_mid:.2f}x")

    # --- Uncertainty-error correlation ---
    corr_ue, p_ue = pearsonr(y_test_std, abs_errors)
    print(f"\n  --- Uncertainty-Error Correlation ---")
    print(f"  Pearson r = {corr_ue:.4f} (p = {p_ue:.2e})")

    # --- Soil moisture extremes ---
    sm_q10 = np.percentile(soil_moisture_vals, 10)
    sm_q90 = np.percentile(soil_moisture_vals, 90)
    sm_extreme_mask = (soil_moisture_vals < sm_q10) | (soil_moisture_vals > sm_q90)

    print(f"\n  --- Soil Moisture Extremes ---")
    print(f"  Extreme SM (<{sm_q10:.3f} or >{sm_q90:.3f}): n={sm_extreme_mask.sum():,} ({100*sm_extreme_mask.mean():.1f}%)")
    print(f"  MAE (extreme SM): {abs_errors[sm_extreme_mask].mean():.4f}")
    print(f"  MAE (normal SM): {abs_errors[~sm_extreme_mask].mean():.4f}")

    # Save results
    regime_results = {
        'mc_dropout_baseline': {
            'mean_sigma': float(y_test_std.mean()),
            'std_sigma': float(y_test_std.std()),
            'median_sigma': float(np.median(y_test_std)),
            'min_sigma': float(y_test_std.min()),
            'max_sigma': float(y_test_std.max()),
            'uncertainty_error_corr': float(corr_ue),
            'uncertainty_error_pval': float(p_ue)
        },
        'valve_regimes': {
            'high_valve_n': int(n_high),
            'high_valve_mae': float(mae_high),
            'high_valve_mean_sigma': float(sigma_high),
            'low_valve_n': int(n_low),
            'low_valve_mae': float(mae_low),
            'low_valve_mean_sigma': float(sigma_low),
            'mae_ratio': float(mae_high / mae_low),
            'sigma_ratio': float(sigma_high / sigma_low),
            'cohens_d': float(cohens_d)
        },
        'humidity_regimes': {
            'top_quartile_threshold': float(humidity_q75),
            'n_humid': int(n_humid),
            'mae_humid': float(mae_humid),
            'error_std_humid': float(std_humid)
        },
        'temperature_regimes': {
            'q25': float(temp_q25),
            'q75': float(temp_q75),
            'mae_low_temp': float(mae_temp_low),
            'mae_mid_temp': float(mae_temp_mid),
            'mae_high_temp': float(mae_temp_high),
            'extreme_vs_mid_ratio': float(mae_temp_extreme / mae_temp_mid)
        }
    }

    with open(TABLES_DIR / 'regime_stats_verified.json', 'w') as f:
        json.dump(regime_results, f, indent=2)

    print(f"\n  Results saved to: {TABLES_DIR / 'regime_stats_verified.json'}")
    return regime_results

## PART 2: Re-compute concentration index on ALL 189K samples (Issue C)

In [ ]:
def part2_concentration_index():
    """Compute correct concentration index (standard Gini) on all samples."""
    print("\n" + "=" * 70)
    print("PART 2: Computing concentration index on ALL 189,534 samples")
    print("=" * 70)

    attention_weights = np.load(RESULTS_DIR / 'attention_weights.npy')
    n_samples, T = attention_weights.shape
    print(f"  Attention weights shape: {attention_weights.shape}")

    # Standard Gini coefficient (matches notebook 05 code)
    # Formula: G = (2 * sum((n - i + 0.5) * sorted_asc[i]) / (n * sum(x))) - 1
    # Range: [0, 1] for standard Gini, but this formulation gives ~ [-0.3, 0] for near-uniform
    print("  Computing Gini coefficients for all samples...")
    t0 = time.time()

    gini_vals = np.zeros(n_samples)
    for i in range(n_samples):
        weights = attention_weights[i]
        sorted_asc = np.sort(weights)
        n = len(sorted_asc)
        gini_vals[i] = (2 * np.sum((n - np.arange(1, n + 1) + 0.5) * sorted_asc)) / (n * np.sum(sorted_asc)) - 1

        if (i + 1) % 50000 == 0:
            print(f"    Processed {i+1:,}/{n_samples:,} samples...")

    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")

    # Statistics
    print(f"\n  --- Concentration Index Statistics (ALL {n_samples:,} samples) ---")
    print(f"  Mean: {gini_vals.mean():.4f}")
    print(f"  Std:  {gini_vals.std():.4f}")
    print(f"  Median: {np.median(gini_vals):.4f}")
    print(f"  Range: [{gini_vals.min():.4f}, {gini_vals.max():.4f}]")

    # Threshold analysis
    high_conc = (gini_vals > 0.5).sum()
    pct_high = 100 * high_conc / n_samples
    print(f"  C_i > 0.5 (high concentration): {high_conc}/{n_samples:,} ({pct_high:.2f}%)")

    # Check range claim from paper: 94.2% in [-0.28, -0.15]
    in_range = ((gini_vals >= -0.28) & (gini_vals <= -0.15)).sum()
    pct_in_range = 100 * in_range / n_samples
    print(f"  In [-0.28, -0.15]: {in_range:,}/{n_samples:,} ({pct_in_range:.1f}%)")

    # Find actual 94.2% range
    pct_025 = np.percentile(gini_vals, 2.9)
    pct_975 = np.percentile(gini_vals, 97.1)
    print(f"  94.2% of data falls in [{pct_025:.4f}, {pct_975:.4f}]")

    # Single-timestep dominance check
    max_attn = np.max(attention_weights, axis=1)
    pathological = (max_attn > 0.8).sum()
    print(f"  Single timestep > 0.8: {pathological}/{n_samples:,} ({100*pathological/n_samples:.2f}%)")

    # KL divergence from uniform
    print("\n  Computing KL divergence from uniform...")
    uniform = np.ones(T) / T
    kl_vals = np.zeros(n_samples)
    for i in range(n_samples):
        kl_vals[i] = np.sum(attention_weights[i] * np.log((attention_weights[i] + 1e-10) / uniform))

    print(f"  Mean KL: {kl_vals.mean():.4f} +/- {kl_vals.std():.4f}")
    print(f"  KL > 0.1 (strongly non-uniform): {(kl_vals > 0.1).sum():,} ({100*(kl_vals > 0.1).mean():.1f}%)")

    # Save results
    conc_results = {
        'n_samples': int(n_samples),
        'gini_mean': float(gini_vals.mean()),
        'gini_std': float(gini_vals.std()),
        'gini_median': float(np.median(gini_vals)),
        'gini_min': float(gini_vals.min()),
        'gini_max': float(gini_vals.max()),
        'pct_high_concentration_gt05': float(pct_high),
        'pct_in_neg028_neg015': float(pct_in_range),
        'range_94pct': [float(pct_025), float(pct_975)],
        'pathological_single_timestep_gt08': int(pathological),
        'kl_mean': float(kl_vals.mean()),
        'kl_std': float(kl_vals.std()),
        'kl_gt01_pct': float(100 * (kl_vals > 0.1).mean())
    }

    with open(TABLES_DIR / 'concentration_index_verified.json', 'w') as f:
        json.dump(conc_results, f, indent=2)

    print(f"\n  Results saved to: {TABLES_DIR / 'concentration_index_verified.json'}")
    return conc_results, gini_vals, kl_vals

## PART 3: Re-generate concentration figure with correct label (Issue E)

In [ ]:
def part3_fix_figures(gini_vals, kl_vals):
    """Regenerate figures with 'Concentration Index' instead of 'Gini Coefficient'."""
    print("\n" + "=" * 70)
    print("PART 3: Regenerating figures with correct 'Concentration Index' labels")
    print("=" * 70)

    attention_weights = np.load(RESULTS_DIR / 'attention_weights.npy')
    y_test = np.load(RESULTS_DIR / 'y_test.npy')
    y_test_pred = np.load(RESULTS_DIR / 'y_test_pred_attention.npy').flatten()

    plt.style.use('seaborn-v0_8-paper')

    # --- Figure: Attention concentration analysis (renamed from Gini) ---
    subsample_size = 20000
    np.random.seed(42)
    sub_idx = np.random.choice(len(gini_vals), subsample_size, replace=False)
    pred_errors_sub = np.abs(y_test[sub_idx] - y_test_pred[sub_idx])
    gini_sub = gini_vals[sub_idx]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Distribution
    axes[0].hist(gini_vals, bins=50, alpha=0.7, color='teal', edgecolor='black')
    axes[0].axvline(gini_vals.mean(), color='red', linestyle='--', linewidth=2,
                    label=f'Mean: {gini_vals.mean():.3f}')
    axes[0].set_xlabel('Concentration Index', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Distribution of Attention Concentration\n(Concentration Index)',
                      fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(axis='y', alpha=0.3)

    # Plot 2: vs Error
    corr_g, p_g = pearsonr(gini_sub, pred_errors_sub)
    axes[1].scatter(gini_sub, pred_errors_sub, alpha=0.3, s=1, c='teal')
    axes[1].set_xlabel('Attention Concentration Index', fontsize=12)
    axes[1].set_ylabel('Prediction Error (MAE)', fontsize=12)
    axes[1].set_title(f'Attention Concentration vs Prediction Error\n({subsample_size:,} samples)',
                      fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].text(0.05, 0.95, f'Correlation: {corr_g:.3f}\np-value: {p_g:.3e}',
                 transform=axes[1].transAxes, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'attention_concentration_analysis.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / 'attention_concentration_analysis.pdf', bbox_inches='tight')
    plt.close()
    print("  Saved: attention_concentration_analysis.png/pdf (label: Concentration Index)")

    # --- Figure: Attention vs Uniform ---
    timesteps = attention_weights.shape[1]
    avg_attention = attention_weights.mean(axis=0)
    uniform_attention = np.ones(timesteps) / timesteps
    timestep_labels = [f't-{timesteps-i}' for i in range(timesteps)]
    x_pos = np.arange(timesteps)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: KL divergence distribution
    axes[0].hist(kl_vals, bins=50, alpha=0.7, color='purple', edgecolor='black')
    axes[0].axvline(kl_vals.mean(), color='red', linestyle='--', linewidth=2,
                    label=f'Mean: {kl_vals.mean():.3f}')
    axes[0].set_xlabel('KL Divergence from Uniform', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Divergence from Uniform Attention Distribution',
                      fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(axis='y', alpha=0.3)

    # Plot 2: Comparison
    width = 0.35
    axes[1].bar(x_pos - width/2, avg_attention, width, label='Learned Attention', alpha=0.8, color='steelblue')
    axes[1].bar(x_pos + width/2, uniform_attention, width, label='Uniform Attention', alpha=0.8, color='coral')
    axes[1].set_xlabel('Timestep', fontsize=12)
    axes[1].set_ylabel('Attention Weight', fontsize=12)
    axes[1].set_title('Learned vs Uniform Attention', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(timestep_labels, rotation=45, ha='right')
    axes[1].legend(fontsize=11)
    axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'attention_vs_uniform.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / 'attention_vs_uniform.pdf', bbox_inches='tight')
    plt.close()
    print("  Saved: attention_vs_uniform.png/pdf")

    print("  All figures regenerated with correct labels.")

## PART 4: Re-compute SHAP bootstrap CIs (Issue B)

In [ ]:
def part4_shap_bootstrap():
    """Re-compute SHAP bootstrap CIs from the temporal importance matrix."""
    print("\n" + "=" * 70)
    print("PART 4: Re-computing SHAP bootstrap CIs")
    print("=" * 70)

    # Check if raw SHAP values exist; if not, we need to re-compute from the model
    shap_temporal_path = TABLES_DIR / 'shap_temporal_importance.csv'
    shap_feature_path = TABLES_DIR / 'shap_feature_importance.csv'

    if not shap_temporal_path.exists():
        print("  ERROR: shap_temporal_importance.csv not found. Cannot re-compute CIs.")
        print("  You need to re-run notebook 03_shap_explainability.ipynb first.")
        return None

    # Load the existing SHAP data
    temporal_df = pd.read_csv(shap_temporal_path, index_col=0)
    feature_df = pd.read_csv(shap_feature_path)

    print(f"  Loaded SHAP temporal importance: {temporal_df.shape}")
    print(f"  Features: {list(temporal_df.index)}")

    # The temporal importance matrix is (features x timesteps) = (6 x 15)
    # Each value is the mean |SHAP| for that feature-timestep pair across 20 samples
    # To compute proper bootstrap CIs, we need the RAW per-sample SHAP values
    # Since those aren't saved, we'll need to re-run the SHAP computation

    # For now, let's re-compute using the saved model and data
    try:
        import shap
        import tensorflow as tf

        print("  Loading model and data for SHAP re-computation...")

        # Load model
        from tensorflow.keras.models import load_model
        sys.path.insert(0, str(BASE_DIR / 'notebooks'))

        # Need to load with custom objects
        model_path = MODELS_DIR / 'attention_lstm_final.h5'
        if not model_path.exists():
            model_path = MODELS_DIR / 'attention_lstm_best.h5'

        # Load the attention-LSTM model
        from tensorflow.keras.layers import Layer

        class AttentionLayer(Layer):
            def __init__(self, units=20, return_sequences=True, **kwargs):
                super(AttentionLayer, self).__init__(**kwargs)
                self.units = units
                self.return_sequences = return_sequences

            def build(self, input_shape):
                self.W = self.add_weight(name='attention_weight',
                    shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)
                self.b = self.add_weight(name='attention_bias',
                    shape=(self.units,), initializer='zeros', trainable=True)
                self.u = self.add_weight(name='attention_vector',
                    shape=(self.units,), initializer='glorot_uniform', trainable=True)
                super(AttentionLayer, self).build(input_shape)

            def call(self, inputs):
                score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
                attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
                attention_weights = tf.expand_dims(attention_weights, axis=-1)
                context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)
                if self.return_sequences:
                    return context_vector, tf.squeeze(attention_weights, axis=-1)
                else:
                    return context_vector

            def get_config(self):
                config = super().get_config()
                config.update({'units': self.units, 'return_sequences': self.return_sequences})
                return config

        model = load_model(str(model_path), custom_objects={'AttentionLayer': AttentionLayer})
        print(f"  Model loaded from: {model_path}")

        # Load test data
        X_test = np.load(RESULTS_DIR / 'X_test.npy')
        X_train = np.load(RESULTS_DIR / 'X_train.npy')

        # Flatten for SHAP KernelExplainer
        n_test, timesteps, n_features = X_test.shape
        X_test_flat = X_test.reshape(n_test, -1)
        X_train_flat = X_train.reshape(len(X_train), -1)

        # Stratified sampling of 20 test instances (same as original)
        np.random.seed(42)
        sample_indices = np.random.choice(n_test, 20, replace=False)
        X_shap = X_test_flat[sample_indices]

        # Background data (100 training samples)
        bg_indices = np.random.choice(len(X_train_flat), 100, replace=False)
        X_background = X_train_flat[bg_indices]

        # Define prediction function for flattened input
        def predict_flat(X_flat):
            X_3d = X_flat.reshape(-1, timesteps, n_features)
            return model.predict(X_3d, verbose=0).flatten()

        # Create SHAP explainer
        print("  Creating SHAP KernelExplainer...")
        explainer = shap.KernelExplainer(predict_flat, X_background)

        # Compute SHAP values for 20 samples
        print("  Computing SHAP values for 20 stratified samples...")
        print("  (This may take 30-60 minutes)")
        t0 = time.time()
        shap_values = explainer.shap_values(X_shap, nsamples=1000, silent=True)
        elapsed = time.time() - t0
        print(f"  SHAP computation done in {elapsed/60:.1f} minutes")

        # Reshape to 3D: (20, 15, 6)
        shap_3d = shap_values.reshape(20, timesteps, n_features)

        # Now bootstrap 1000 times
        print("  Running bootstrap (1000 iterations)...")
        n_boot = 1000
        n_shap_samples = 20

        import pickle
        with open(RESULTS_DIR / 'feature_names.pkl', 'rb') as f:
            feat_names = pickle.load(f)

        boot_feature_importance = np.zeros((n_boot, n_features))

        for b in range(n_boot):
            boot_idx = np.random.choice(n_shap_samples, n_shap_samples, replace=True)
            boot_shap = shap_3d[boot_idx]
            boot_feature_importance[b] = np.abs(boot_shap).mean(axis=(0, 1))

        # Compute CIs
        ci_results = []
        for f_idx, fname in enumerate(feat_names):
            mean_imp = np.abs(shap_3d[:, :, f_idx]).mean()
            ci_lower = np.percentile(boot_feature_importance[:, f_idx], 2.5)
            ci_upper = np.percentile(boot_feature_importance[:, f_idx], 97.5)
            ci_results.append({
                'Feature': fname,
                'Mean_Importance': float(mean_imp),
                'CI_Lower': float(ci_lower),
                'CI_Upper': float(ci_upper),
                'CI_Width': float(ci_upper - ci_lower)
            })
            print(f"  {fname}: {mean_imp:.6f} [{ci_lower:.6f}, {ci_upper:.6f}]")

        # Save corrected CIs
        ci_df = pd.DataFrame(ci_results)
        ci_df.to_csv(TABLES_DIR / 'shap_importance_with_ci.csv', index=False)
        print(f"\n  Saved corrected CIs to: {TABLES_DIR / 'shap_importance_with_ci.csv'}")

        # Also save raw SHAP values for future use
        np.save(RESULTS_DIR / 'shap_values_raw_20samples.npy', shap_3d)
        print(f"  Saved raw SHAP values to: {RESULTS_DIR / 'shap_values_raw_20samples.npy'}")

        return ci_results

    except Exception as e:
        print(f"  WARNING: SHAP re-computation failed: {e}")
        print("  You may need to re-run notebook 03_shap_explainability.ipynb manually.")
        print("  The feature importance values are verified correct from shap_feature_importance.csv")
        return None

## PART 5: Train 8-seed attention-LSTM on full data (Issue A)

In [ ]:
def part5_eight_seed_attention():
    """Train 8 attention-LSTM models on full data with different seeds."""
    print("\n" + "=" * 70)
    print("PART 5: Training 8-seed attention-LSTM on FULL data")
    print("=" * 70)
    print("  WARNING: This will take approximately 4 hours on CPU!")

    import tensorflow as tf
    from tensorflow.keras import layers, Model, Input
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    # Attention layer definition
    class AttentionLayer(layers.Layer):
        def __init__(self, units=20, return_sequences=True, **kwargs):
            super(AttentionLayer, self).__init__(**kwargs)
            self.units = units
            self.return_sequences = return_sequences

        def build(self, input_shape):
            self.W = self.add_weight(name='attention_weight',
                shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)
            self.b = self.add_weight(name='attention_bias',
                shape=(self.units,), initializer='zeros', trainable=True)
            self.u = self.add_weight(name='attention_vector',
                shape=(self.units,), initializer='glorot_uniform', trainable=True)
            super(AttentionLayer, self).build(input_shape)

        def call(self, inputs):
            score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
            attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
            attention_weights = tf.expand_dims(attention_weights, axis=-1)
            context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)
            if self.return_sequences:
                return context_vector, tf.squeeze(attention_weights, axis=-1)
            else:
                return context_vector

        def get_config(self):
            config = super().get_config()
            config.update({'units': self.units, 'return_sequences': self.return_sequences})
            return config

    def build_attention_lstm(input_shape, lstm_units=20, attention_units=20):
        inputs = Input(shape=input_shape, name='input_sequences')
        lstm_output = layers.LSTM(lstm_units, return_sequences=True, name='lstm_layer')(inputs)
        context_vector, attn_weights = AttentionLayer(units=attention_units, return_sequences=True, name='attention_layer')(lstm_output)
        outputs = layers.Dense(1, name='output_layer')(context_vector)
        model = Model(inputs=inputs, outputs=outputs, name='attention_lstm')
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        return model

    # Load data
    print("  Loading data...")
    X_train = np.load(RESULTS_DIR / 'X_train.npy')
    y_train = np.load(RESULTS_DIR / 'y_train.npy')
    X_test = np.load(RESULTS_DIR / 'X_test.npy')
    y_test = np.load(RESULTS_DIR / 'y_test.npy')

    timesteps, n_features = X_train.shape[1], X_train.shape[2]
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")

    # 8 seeds (same as baseline experiment)
    SEEDS = [42, 123, 456, 789, 1011, 1213, 1415, 1617]

    results = []

    for i, seed in enumerate(SEEDS):
        print(f"\n  --- Seed {seed} ({i+1}/{len(SEEDS)}) ---")
        t0 = time.time()

        # Set seeds
        np.random.seed(seed)
        tf.random.set_seed(seed)

        # Build model
        model = build_attention_lstm((timesteps, n_features))

        # Split validation from training (last 10%)
        val_size = int(len(X_train) * 0.1)
        X_t = X_train[:-val_size]
        y_t = y_train[:-val_size]
        X_v = X_train[-val_size:]
        y_v = y_train[-val_size:]

        # Callbacks
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        # Train
        history = model.fit(
            X_t, y_t,
            validation_data=(X_v, y_v),
            epochs=50,
            batch_size=80,
            callbacks=[early_stop],
            verbose=0
        )

        # Evaluate on test set
        y_pred = model.predict(X_test, verbose=0).flatten()
        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))

        elapsed = time.time() - t0
        converged = r2 > 0

        results.append({
            'Seed': seed,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2,
            'Epochs': len(history.history['loss']),
            'Converged': converged
        })

        status = "CONVERGED" if converged else "FAILED"
        print(f"    {status}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f} ({elapsed:.0f}s)")

        # Clear model to free memory
        del model
        tf.keras.backend.clear_session()

    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv(TABLES_DIR / 'attention_8seed_fulldata.csv', index=False)

    # Print summary
    print(f"\n  --- 8-Seed Attention-LSTM Summary (Full Data) ---")
    n_converged = results_df['R2'].apply(lambda x: x > 0).sum()
    n_gt05 = results_df['R2'].apply(lambda x: x > 0.5).sum()
    n_gt07 = results_df['R2'].apply(lambda x: x > 0.7).sum()

    print(f"  R2 > 0.0: {n_converged}/8 ({100*n_converged/8:.1f}%)")
    print(f"  R2 > 0.5: {n_gt05}/8 ({100*n_gt05/8:.1f}%)")
    print(f"  R2 > 0.7: {n_gt07}/8 ({100*n_gt07/8:.1f}%)")

    converged_r2 = results_df[results_df['R2'] > 0]['R2']
    if len(converged_r2) > 0:
        print(f"  Mean R2 (converged): {converged_r2.mean():.4f} +/- {converged_r2.std():.4f}")
        print(f"  Best R2: {results_df['R2'].max():.4f}")
        print(f"  Worst converged R2: {converged_r2.min():.4f}")

    print(f"\n  Results saved to: {TABLES_DIR / 'attention_8seed_fulldata.csv'}")

    return results_df

## MAIN

In [ ]:
if __name__ == '__main__':
    all_results = {}

    # Part 1: Quick - regime stats (~30s)
    all_results['regime_stats'] = part1_regime_stats()

    # Part 2: Quick - concentration index (~2 min)
    conc_results, gini_vals, kl_vals = part2_concentration_index()
    all_results['concentration_index'] = conc_results

    # Part 3: Quick - regenerate figures (~10s)
    part3_fix_figures(gini_vals, kl_vals)

    # Part 4: Medium - SHAP bootstrap (~30-60 min)
    all_results['shap_cis'] = part4_shap_bootstrap()

    # Part 5: Long - 8-seed attention training (~4 hours)
    all_results['eight_seed_attention'] = part5_eight_seed_attention()

    # Save comprehensive results
    print("\n" + "=" * 70)
    print("ALL DONE! Summary of verified data:")
    print("=" * 70)

    # Print key numbers for manuscript
    rs = all_results['regime_stats']
    print(f"\nMC-Dropout Baseline:")
    print(f"  Mean sigma: {rs['mc_dropout_baseline']['mean_sigma']:.4f}")
    print(f"  Valve regime MAE ratio: {rs['valve_regimes']['mae_ratio']:.1f}x")
    print(f"  High valve MAE: {rs['valve_regimes']['high_valve_mae']:.4f}")
    print(f"  Low valve MAE: {rs['valve_regimes']['low_valve_mae']:.4f}")

    ci = all_results['concentration_index']
    print(f"\nConcentration Index:")
    print(f"  Mean: {ci['gini_mean']:.4f}")
    print(f"  Std: {ci['gini_std']:.4f}")
    print(f"  94.2% range: {ci['range_94pct']}")

    if all_results.get('eight_seed_attention') is not None:
        df = all_results['eight_seed_attention']
        print(f"\n8-Seed Attention (Full Data):")
        for _, row in df.iterrows():
            print(f"  Seed {int(row['Seed'])}: R2={row['R2']:.4f}")

    print("\nAll results saved to results/tables/")
    print("Please update the LaTeX manuscript with these verified numbers.")